In [1]:
import json
import torch
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-4")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-4",
    torch_dtype="auto",
    device_map="auto"
)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [4]:
# import transformers

# pipeline = transformers.pipeline(
#     "text-generation",
#     model="microsoft/phi-4",
#     model_kwargs={"torch_dtype": "auto"},
#     device_map="auto",
# )


In [4]:
messages = [
    {"role": "system", "content": "You are a medieval knight and must provide explanations to modern people."},
    {"role": "user", "content": "How should I explain the Internet?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Ah, greetings, traveler from the future! Thou art inquiring about a wondrous and mystical realm known as the Internet. Allow me to endeavor to explain this marvel in terms that might be more familiar


In [ ]:
# from unsloth import FastLanguageModel  # FastVisionModel for LLMs
# import torch
# MAX_SEQ_LENGTH = 8196  # Choose any! We auto support RoPE Scaling internally!
# load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.


# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Phi-4",
#     max_seq_length = MAX_SEQ_LENGTH,
#     load_in_4bit = load_in_4bit,
#     # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
# )

In [ ]:
# messages = [
#     {"role": "user", "content": "Who are you?"},
# ]
# inputs = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
# ).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=40)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

I am Phi, a language model developed by Microsoft. My purpose is to assist users by providing information, answering questions, and helping with a wide range of topics to the best of my ability. If


In [5]:
print(model.device)

cuda:0


In [6]:
# Few-shot template 
FEW_SHOT_HINDI = """
# Instruction:
You are a helpful assistant who generates answers from a Hindi table to answer Hindi questions.  
Use the below example to guide the format. 

## Example:

### Input:
 ले ट्रुनिन ने किन फिल्मों में भूमिका निभाई थी?  

<column>  वर्ष | शीर्षक | भूमिका  
<row 1> 2014 | See No Evil 2 | जैकब गुडनाइट  
<row 2> 2016 | Countdown | ले ट्रुनिन  
<row 3> 2017 | Meltdown | ले ट्रुनिन  

### Response (complete this):
<column> शीर्षक  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_TELUGU = """
# Instruction:
You are a helpful assistant who generates answers from a Telugu table to answer Telugu questions.  
Use the below example to guide the format. 

## Example:

### Input:
 లే ట్రునిన్ ఏ సినిమాలలో పాత్ర పోషించాడు?

<column>  సంవత్సరం | శీర్షిక | పాత్ర  
<row 1> 2014 | See No Evil 2 | జేకబ్ గుడ్‌నైట్  
<row 2> 2016 | Countdown | లే ట్రునిన్  
<row 3> 2017 | Meltdown | లే ట్రునిన్  

### Response (complete this):
<column> శీర్షిక  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

FEW_SHOT_BENGALI = """
# Instruction:
You are a helpful assistant who generates answers from a Bengali table to answer Bengali questions.  
Use the below example to guide the format. 

## Example:

### Input:
 লে ট্রুনিন কোন কোন সিনেমায় অভিনয় করেছেন?

<column>  বছর | শিরোনাম | চরিত্র  
<row 1> 2014 | See No Evil 2 | জ্যাকব গুডনাইট  
<row 2> 2016 | Countdown | লে ট্রুনিন  
<row 3> 2017 | Meltdown | লে ট্রুনিন  

### Response (complete this):
<column> শিরোনাম  
<row 1> Countdown  
<row 2> Meltdown  

now answer the following question, generate only the ouput table and nothing else.
"""

def table_to_text(table):
    if not table:
        return ""
    columns = list(table[0].keys())
    header = " | ".join(columns)
    rows = []
    for i, row in enumerate(table):
        row_str = " | ".join(str(row[col]) for col in columns)
        rows.append(f"<row {i+1}> {row_str}")
    return f"<column>\n{header}\n" + "\n".join(rows)

def resultdf_to_text(result_df):
    if not result_df:
        return ""
    df = pd.DataFrame(result_df)
    return df.to_string(index=False)

In [7]:
def run_tableqa_testset_phi(model, tokenizer, model_instruction, testset_path, result_path, sanity=False):
    """
    Loads a test set, prompts the model for each query, and writes results.
    If sanity=True, processes only one question and saves in sanity_ file.
    """

    generation_config = {"max_new_tokens": 1024}

    # Load test data
    with open(testset_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    # Determine tables/queries to process
    tables_to_process = test_data if not sanity else [test_data[0]]
    total_queries = 1 if sanity else sum(len(t["queries"]) for t in tables_to_process)

    with tqdm(total=total_queries, desc="Processing queries") as pbar:
        for table in tables_to_process:

            full_table = table["full_table_df"]
            table_text = table_to_text(full_table)

            queries = [table["queries"][0]] if sanity else table["queries"]

            for q in queries:
                question = q["question"]

                # Build prompt
                prompt = (
                    f"\n###Input:\n{question}\n{table_text}\n###Response:\n"
                )

                messages = [
                    {"role": "system", "content": model_instruction},
                    {"role": "user", "content": prompt},
                ]

                # -------------------------------
                #  GPU-SAFE INFERENCE
                # -------------------------------
                with torch.no_grad():

                    # Tokenization → move to GPU
                    inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt"
                    ).to(model.device)

                    # Generation without cache
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=generation_config["max_new_tokens"],
                    )

                    # Decode only new tokens
                    response = tokenizer.decode(
                        outputs[0][inputs["input_ids"].shape[-1]:],
                        skip_special_tokens=True
                    )

                # Extract answer after ###Response:
                if "###Response:" in response:
                    response = response.split("###Response:")[-1].strip()

                # Save result
                q["model_response"] = {
                    "content": response,
                    "thinking_content": ""
                }

                # Cleanup GPU memory
                del inputs
                del outputs

                pbar.update(1)

                # Sanity mode: print + write one result then exit
                if sanity:
                    print("Prompt:\n", prompt)
                    print("Response:\n", response)

                    dir_path = os.path.dirname(result_path)
                    base_name = os.path.basename(result_path)
                    sanity_path = os.path.join(dir_path, "sanity_" + base_name)

                    with open(sanity_path, "w", encoding="utf-8") as f:
                        json.dump([table], f, ensure_ascii=False, indent=2)
                    return

    # Final write (normal mode)
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)

In [11]:
run_tableqa_testset_phi(
    model=model,
    tokenizer=tokenizer,
    model_instruction=FEW_SHOT_BENGALI,
    testset_path="data/bengali/bengali_testset.json",
    result_path="data/bengali/phi4_results_1.json",
    sanity=False
)

Processing queries:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing queries: 100%|██████████| 1000/1000 [2:36:11<00:00,  9.37s/it] 
